Project name : Group Dna

Done by : Anona Mannali Ratheesh

Data Science (june 2026 - batch)

# 1.Chat Parser
Reads the exported file and converts every valid message into a structured dictionary


* messages stores all the chat messages
* partcipants store the name of members uniquely
* system_messages, media_omitted, deleted_messages are also counted
* file.readlines() reads line one by one
* if line starts with a date is treated as anew message else treated as continuation of previous message
* parser splits a message into 3 parts - timestamp, sender, message
* detect system message, if ':' not there
* splits sender and messages by ':'
* if only first line contains timestamp and others a continuation by same sender, parser identifies them as continuos message and appends them to last_message




In [ ]:
messages = []

system_messages = 0
media_omitted = 0
deleted_messages = 0

participants = set()

with open("hostel_bois.txt", "r", encoding="utf-8") as file:
    lines = file.readlines()

last_message = None

for line in lines:
    line = line.strip()

    # Skip empty lines
    if line == "":
        continue

    # Check whether line starts with a date
    if len(line) >= 8 and line[0:2].isdigit() and line[2] == "/" and line[3:5].isdigit() and line[5] == "/" and line[6:8].isdigit():

        # Split timestamp from remaining part
        if " - " not in line:
            continue
        parts = line.split(" - ", 1)
        timestamp = parts[0]
        remaining = parts[1]

        # System message has no ": "
        if ": " not in remaining:
            system_messages += 1
            continue

        # Split sender and message
        sender, text = remaining.split(": ", 1)

        # Count media omitted
        if text == "<Media omitted>":
            media_omitted += 1

        # Count deleted messages
        if text == "This message was deleted":
            deleted_messages += 1

        # Store message
        message = {
            "timestamp": timestamp,
            "sender": sender,
            "text": text
        }
        messages.append(message)
        participants.add(sender)
        last_message = message

    else:
        # Multi-line message handle
        if last_message is not None:
            last_message["text"] += "\n" + line


print("=" * 60)
print("CHAT PARSER")
print("=" * 60)

print("Total Parsed Messages :", len(messages))
print("Participants          :", len(participants))
print("System Messages       :", system_messages)
print("Media Omitted         :", media_omitted)
print("Deleted Messages      :", deleted_messages)

print("\nParticipants List")
for person in sorted(participants):
    print("-", person)

print("\nSuccessfully parsed", len(messages), "messages from",
      len(participants), "participants.")

CHAT PARSER
Total Parsed Messages : 3174
Participants          : 6
System Messages       : 4
Media Omitted         : 32
Deleted Messages      : 15

Participants List
- Aman
- Karan
- Neha
- Priya
- Rahul
- Vikas

Successfully parsed 3174 messages from 6 participants.


# 2.Group overview
Overview of the group, calculates total messages, total participation, chat duration, messages sent, percent of message sent by each member.
* datetime function is used to convert date strings to date object for calculation.
* message_count stores how many message each participant has sent
* date_count stores how many messages were sent on a day, date is extracted from timestamp for this.
* dates are converted into datetime objects and sorted by datetime.strptime()
* first and last chats are found out and their difference + 1 is the duration of the group.


In [ ]:
from datetime import datetime

message_count = {}
date_count = {}

# Count messages
for msg in messages:
    sender = msg["sender"]
    timestamp = msg["timestamp"]

    # Count messages of each person
    if sender in message_count:
        message_count[sender] += 1
    else:
        message_count[sender] = 1

    # Extract date alone
    date = timestamp.split(",")[0]

    if date in date_count:
        date_count[date] += 1
    else:
        date_count[date] = 1


# Total messages
total_messages = len(messages)

# Total participants
total_participants = len(participants)

# Find first and last dates
date_list = list(date_count.keys())
date_objects = []

for d in date_list:
    date_objects.append(datetime.strptime(d, "%d/%m/%y"))

date_objects.sort()
start_date = date_objects[0]
end_date = date_objects[-1]

# Total days
total_days = (end_date - start_date).days + 1

# Sort participants by message count
sorted_people = sorted(message_count.items(),key=lambda x: x[1],reverse=True)

print("=" * 60)
print("GROUP OVERVIEW")
print("=" * 60)

print(f"Group             : Hostel Bois 4ever")
print(f"Period            : {start_date.strftime('%d %B %Y')} to {end_date.strftime('%d %B %Y')} ({total_days} days)")
print(f"Total Messages    : {total_messages}")
print(f"Participants      : {total_participants}")

print("\nMESSAGES PER PERSON")
print("-" * 60)

for person, count in sorted_people:

    percentage = (count / total_messages) * 100

    print(f"{person:<10} : {count:>4} ({percentage:.1f}%)")

GROUP OVERVIEW
Group             : Hostel Bois 4ever
Period            : 01 April 2024 to 30 May 2024 (60 days)
Total Messages    : 3174
Participants      : 6

MESSAGES PER PERSON
------------------------------------------------------------
Rahul      :  953 (30.0%)
Priya      :  718 (22.6%)
Neha       :  635 (20.0%)
Aman       :  490 (15.4%)
Karan      :  354 (11.2%)
Vikas      :   24 (0.8%)


# 3. Most active Day and Hour
Finds the busiest day and hour in the group by number of messages sent.
* day_count similar to date_count in feature2 but used here to find busiest day
* day_count stores number of messages sent on each date and hour_count number of messages in an hour.
* datetime converted from string to object and extracted only date and hour for each and counted them into date_count and hour_count.
* busiest day, found out using comparing message count of each day, day having highest message count is busiest and similary busiest hour is also calculated'
* strftime() used to convert date into readable format

In [ ]:
day_count = {}

hour_count = {}

for msg in messages:
    timestamp = msg["timestamp"]

    # Convert timestamp into datetime object
    dt = datetime.strptime(timestamp, "%d/%m/%y, %H:%M")

    # Date (DD/MM/YY)
    date = dt.date()

    # Hour (0-23)
    hour = dt.hour

    # Count messages per day
    if date in day_count:
        day_count[date] += 1
    else:
        day_count[date] = 1

    # Count messages per hour
    if hour in hour_count:
        hour_count[hour] += 1
    else:
        hour_count[hour] = 1

# Find busiest day

busiest_day = None
busiest_day_messages = 0

for day in day_count:

  if day_count[day] > busiest_day_messages:
    busiest_day = day
    busiest_day_messages = day_count[day]

# Find busiest hour

busiest_hour = None
busiest_hour_messages = 0

for hour in hour_count:

  if hour_count[hour] > busiest_hour_messages:
    busiest_hour = hour
    busiest_hour_messages = hour_count[hour]

print("=" * 60)
print("MOST ACTIVE DAY AND HOUR")
print("=" * 60)

print(f"Busiest Day  : {busiest_day.strftime('%d %B %Y')} ({busiest_day_messages} messages)")

print(f"Busiest Hour : {busiest_hour:02d}:00 - {(busiest_hour + 1) % 24:02d}:00 ({busiest_hour_messages} messages)")

MOST ACTIVE DAY AND HOUR
Busiest Day  : 04 May 2024 (76 messages)
Busiest Hour : 18:00 - 19:00 (248 messages)


# 4. Activity Heatmap
To observe and analyse when each participant is active during the day. Creates a matrix where rows = participants and columns = hours
* 'partcipants' which was set before is now converted into list and sorted alphabetically .
* Row indexing done to give each participant a fixed row number in the heatmap.
* numpy matrix created and every value initially is 0 because no message is counted yet.
* Every message is increments the corresponding cell based on the sender and the hour it was sent.
* Heatmap displayed with shaded bars to observe activity levels.
* np.sum is used to calculate total messages sent by each participant and total message during each hour.

In [ ]:
import numpy as np

# Convert participants set to sorted list
people = sorted(list(participants))

# Create a dictionary for row indexing
person_index = {}

for i in range(len(people)):
    person_index[people[i]] = i

# Create a 6 x 24 matrix
heatmap = np.zeros((len(people), 24), dtype=int)

# Fill the matrix
for msg in messages:
    sender = msg["sender"]
    dt = datetime.strptime(msg["timestamp"], "%d/%m/%y, %H:%M")
    hour = dt.hour
    row = person_index[sender]
    heatmap[row][hour] += 1

# Display numpy matrix

print("=" * 70)
print("NUMPY ACTIVITY MATRIX")
print("=" * 70)

print(heatmap)

# Print text heatmap

print("\n")
print("=" * 70)
print("ACTIVITY HEATMAP (Messages by Hour)")
print("=" * 70)

print("Name      ", end="")

for h in range(24):
    print(f"{h:02d} ", end="")

print()

# Print heatmap row by row
for i in range(len(people)):
    print(f"{people[i]:<10}", end="")
    row = heatmap[i]
    maximum = np.max(row)

    if maximum == 0:
        maximum = 1

    for value in row:
        ratio = value / maximum

        if value == 0:
            symbol = ". "
        elif ratio <= 0.25:
            symbol = "\u2591 "
        elif ratio <= 0.50:
            symbol = "\u2592 "
        elif ratio <= 0.75:
            symbol = "\u2593 "
        else:
            symbol = "\u2588 "

        print(symbol, end="")

    print()

# TOTAL MESSAGES BY PERSON

print("\n")
print("=" * 70)
print("TOTAL MESSAGES BY PERSON (Using NumPy)")
print("=" * 70)

for i in range(len(people)):
    total = np.sum(heatmap[i])
    print(f"{people[i]:<10} : {total}")

# TOTAL MESSAGES EACH HOUR

print("\n")
print("=" * 70)
print("TOTAL MESSAGES EACH HOUR")
print("=" * 70)

hour_total = np.sum(heatmap, axis=0)

for h in range(24):
    print(f"{h:02d}:00 - {h:02d}:59 : {hour_total[h]}")

NUMPY ACTIVITY MATRIX
[[ 54  67  66  60  88   0   0   0   0   0   0   0   0   0  14  11  19   7
   16   8  13  11   0  56]
 [  0   0   0   0   0   0   0   4  12  16  20  16  37  25  32  27  27  27
   25  32  23  14   9   8]
 [  0   0   0   0   0  19   3  13  36  52  52  22  39  36  27  10  37  47
   62  50  45  27  28  30]
 [  0   0   0   0   0   0  13  20  47  65  62  61  57  48  44  29  32  40
   38  60  43  32  18   9]
 [  3  15  17  17  22  10  17  17  24  17  25  15  58  48  45  53  73  49
  105  76  41  92  60  54]
 [  0   0   0   0   0   0   0   1   3   1   1   0   2   2   0   1   1   3
    2   2   1   1   1   2]]


ACTIVITY HEATMAP (Messages by Hour)
Name      00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23 
Aman      ▓ █ ▓ ▓ █ . . . . . . . . . ░ ░ ░ ░ ░ ░ ░ ░ . ▓ 
Karan     . . . . . . . ░ ▒ ▒ ▓ ▒ █ ▓ █ ▓ ▓ ▓ ▓ █ ▓ ▒ ░ ░ 
Neha      . . . . . ▒ ░ ░ ▓ █ █ ▒ ▓ ▓ ▒ ░ ▓ █ █ █ ▓ ▒ ▒ ▒ 
Priya     . . . . . . ░ ▒ ▓ █ █ █ █ ▓ ▓ ▒ ▒ ▓ ▓ █ ▓ ▒ ▒ ░ 
Rahul     ░ ░ 

# 5. Top 5 words
Finds the most frequently used words by removing some common words.
* word_count stores each unique words, stop_words contain common english words which are to be excluded, punctuations are removed
* by processing every message one by one from messages list, media omitted and deleted messages are skipped.
* each message is converted to lowercase, punctuations are replaced with spaces. split() divides each cleaned message into individual words.Empty words and stop words skipped.
* Every remaining words are counted into word_count
* dictionary sorted in descending order using sorted()
* Highest frequency used for bars to visualize

In [ ]:
word_count = {}

# Stop words
stop_words = [
    "i", "is", "the", "a", "an", "and", "or", "to", "of",
    "in", "on", "for", "it", "its", "am", "are", "was",
    "were", "be", "been", "this", "that", "you", "your",
    "me", "my", "we", "our", "us", "he", "she", "they",
    "them", "their", "at", "by", "with", "from", "as",
    "have", "has", "had", "do", "does", "did", "will",
    "would", "can", "could", "should", "if", "but", "how",
    "so", "which", "about", "his","s","just","today","up"
    ]

# Characters to remove
punctuation = ".,!?;:'\"()[]{}<>/-_*&%^$#@+=|\\`~"

# Process every message
for msg in messages:
    text = msg["text"]

    # skip media omitted
    if text == "<Media omitted>":
        continue

    # skip deleted messages
    if text == "This message was deleted":
        continue

    # convert to lowercase
    text = text.lower()

    # remove punctuation
    for ch in punctuation:
        text = text.replace(ch, " ")

    # split into words
    words = text.split()

    # count words
    for word in words:
        word = word.strip()

        if word == "":
            continue
        if word in stop_words:
            continue
        if word in word_count:
            word_count[word] += 1
        else:
            word_count[word] = 1

# SORT WORDS

sorted_words = sorted(word_count.items(),key=lambda x: x[1],reverse=True)
top_words = sorted_words[:5]


print("=" * 60)
print("THIS GROUP'S FAVOURITE WORDS")
print("=" * 60)

# Find highest frequency for scaling bars
highest = top_words[0][1]

for word, count in top_words:
    bar_length = int((count / highest) * 20)
    bar = "\u2588" * bar_length
    print(f"{word:<15} {bar:<20} {count}")

THIS GROUP'S FAVOURITE WORDS
guys            ████████████████████ 318
hai             ████████████████     268
everyone        ████████████         203
telling         ███████████          179
bhai            ██████████           160


# 6. Response Speed & Silent Streaks
Average response time and Longest silent streak is calculated.
* transverse message list from second message onwards for every consecutive messages it checks if the sender has changed.
* If both messages are from the same sender its ignored otherwise both timestamps conerted into datetime objects.
* Time diff between 2 messages calculated this is stored as response time under current sender.
* after all messages are processed, avg response time of every member is calculated (which is also provide in the output)
* member with minimum average is fast_person and maximum avg slowest_person.
* Longest Silent Streak
* active_days stores all dates on which each paricipant sent atleast one message.
* participants value is a set so no duplicates
* For every participant, checks every day between start date and end date, if partcipant has no message on oneday the silent streak increases if they send a message current streak becomes 0
* if current streak > previous maximum it becomes lowest streak.

In [ ]:
from datetime import datetime, timedelta

# Average response time

# to store response times (in seconds)
response_time = {}

# Traverse messages
for i in range(1, len(messages)):
    previous = messages[i - 1]
    current = messages[i]

    # Ignore consecutive messages from same sender
    if previous["sender"] == current["sender"]:
        continue

    # Convert timestamps into datetime objects
    previous_time = datetime.strptime(previous["timestamp"], "%d/%m/%y, %H:%M")
    current_time = datetime.strptime(current["timestamp"], "%d/%m/%y, %H:%M")

    # Calculate time difference in seconds
    gap = (current_time - previous_time).total_seconds()
    sender = current["sender"]

    # Store response time
    if sender not in response_time:
        response_time[sender] = []

    response_time[sender].append(gap)

# calculate avg response time

average_response = {}

for person in response_time:
    total = sum(response_time[person])
    count = len(response_time[person])
    average = total / count
    average_response[person] = average

# Longest silent streak

# to store active dates of each participant
active_days = {}

# Initialize dictionary
for person in participants:
    active_days[person] = set()

# Store active dates
for msg in messages:
    person = msg["sender"]
    date = datetime.strptime(msg["timestamp"], "%d/%m/%y, %H:%M").date()
    active_days[person].add(date)

# Find overall chat date range
start_date = datetime.strptime(messages[0]["timestamp"], "%d/%m/%y, %H:%M").date()
end_date = datetime.strptime(messages[-1]["timestamp"], "%d/%m/%y, %H:%M").date()

print("=" * 60)
print("AVERAGE RESPONSE TIME")
print("=" * 60)

fastest_person = ""
slowest_person = ""

fastest_time = float("inf")
slowest_time = 0

for person in average_response:
    seconds = average_response[person]

    if seconds < fastest_time:
        fastest_time = seconds
        fastest_person = person

    if seconds > slowest_time:
        slowest_time = seconds
        slowest_person = person

    if seconds < 3600:
        value = seconds / 60
        unit = "minutes"
    else:
        value = seconds / 3600
        unit = "hours"

    print(f"{person:<10} : {value:.2f} {unit}")

print("\nFastest Replier :", fastest_person)

if fastest_time < 3600:
    print(f"Average Time    : {fastest_time/60:.2f} minutes")
else:
    print(f"Average Time    : {fastest_time/3600:.2f} hours")

print()

print("Slowest Replier :", slowest_person)

if slowest_time < 3600:
    print(f"Average Time    : {slowest_time/60:.2f} minutes")
else:
    print(f"Average Time    : {slowest_time/3600:.2f} hours")

print("=" * 60)
print("LONGEST SILENT STREAKS")
print("=" * 60)

# to store longest streak of each participant
longest_streaks = {}

# Calculate streak for each participant
for person in sorted(participants):
    longest_streak = 0
    current_streak = 0

    longest_start = None
    longest_end = None

    current_start = None
    current_date = start_date

    while current_date <= end_date:

        if current_date not in active_days[person]:
            if current_streak == 0:
                current_start = current_date
            current_streak += 1
            if current_streak > longest_streak:
                longest_streak = current_streak
                longest_start = current_start
                longest_end = current_date
        else:
            current_streak = 0
        current_date += timedelta(days=1)

        # to save streak in dictionary
        longest_streaks[person] = longest_streak

        # result
    if longest_streak == 0:
        print(f"{person:<10} : 0 days")
    else:
        print(f"{person:<10} : {longest_streak} days "
              f"({longest_start.strftime('%d %b')} - {longest_end.strftime('%d %b')})")


AVERAGE RESPONSE TIME
Priya      : 41.99 minutes
Rahul      : 34.95 minutes
Karan      : 36.62 minutes
Neha       : 39.45 minutes
Aman       : 55.36 minutes
Vikas      : 46.30 minutes

Fastest Replier : Rahul
Average Time    : 34.95 minutes

Slowest Replier : Aman
Average Time    : 55.36 minutes
LONGEST SILENT STREAKS
Aman       : 0 days
Karan      : 0 days
Neha       : 0 days
Priya      : 0 days
Rahul      : 0 days
Vikas      : 11 days (23 Apr - 03 May)


# Personality Archetypes

The spammer:

Participant who sends longest consecutive message bursts
spammer_data stores consecutive message burst length for each participant.
* calculates average burst length of every participant
* sort participants based on their burst length
* participant with highest average is the spammer
(runner up is also found out for every archetypes)






In [ ]:
spammer_data = {}

for person in participants:
    spammer_data[person] = []

current_sender = messages[0]["sender"]
burst = 1

# Find consecutive message bursts
for i in range(1, len(messages)):

    if messages[i]["sender"] == current_sender:
        burst += 1
    else:
        spammer_data[current_sender].append(burst)
        current_sender = messages[i]["sender"]
        burst = 1

# Store last burst
spammer_data[current_sender].append(burst)

# Calculate average burst length
spammer_score = {}

for person in spammer_data:
    bursts = spammer_data[person]

    if len(bursts) == 0:
        average = 0
    else:
        average = sum(bursts) / len(bursts)
    spammer_score[person] = average

# Sort by score (Highest First)
sorted_spammers = sorted(spammer_score.items(),key=lambda x: x[1],reverse=True)

# Winner
spammer = sorted_spammers[0][0]
spammer_value = sorted_spammers[0][1]

# Runner-up
spammer_runner = sorted_spammers[1][0]
spammer_runner_value = sorted_spammers[1][1]

Group Mom:

counts caring words each participants used
* score stores caring word count for each participant
* converted into lowercase, removed punctuations and split() to split into words
* partcipants sorted in descending order based on their caring_words
* highest group_mom_score winner

In [ ]:
caring_words = [
    "eat", "food", "breakfast", "lunch", "dinner",
    "sleep", "rest", "study", "water", "drink",
    "medicine", "take", "care", "safe", "okay",
    "ok", "fine", "thanks", "thank", "welcome",
    "help", "good", "best", "home", "healthy",
    "congrats", "congratulation", "happy", "don't",
    "dont", "please", "come", "reach", "call"
]

group_mom_score = {}

for person in participants:
    group_mom_score[person] = 0

# Count caring words
for msg in messages:
    sender = msg["sender"]
    text = msg["text"].lower()

    # Remove punctuation
    punctuation = ".,!?;:'\"()[]{}<>/-_*&%^$#@+=|\\`~"

    for ch in punctuation:
        text = text.replace(ch, " ")
    words = text.split()

    for word in words:

        if word in caring_words:
            group_mom_score[sender] += 1

# Sort scores
sorted_group_mom = sorted(group_mom_score.items(),key=lambda x: x[1],reverse=True)

# Winner
group_mom = sorted_group_mom[0][0]
group_mom_value = sorted_group_mom[0][1]

# Runner-up
group_mom_runner = sorted_group_mom[1][0]
group_mom_runner_value = sorted_group_mom[1][1]

Night Owl:

night_messages store number of messages sent at night by each participant ( 12 am to 5.59 am)
* total_messagges stores total number of message sent by each participant.
* percent of message sent at night of each participant stored
* then sorts and determines the winner and runnerup

In [ ]:
night_messages = {}
total_messages = {}

for person in participants:
    night_messages[person] = 0
    total_messages[person] = 0

# Count messages
for msg in messages:
    sender = msg["sender"]
    total_messages[sender] += 1
    time = datetime.strptime(msg["timestamp"], "%d/%m/%y, %H:%M")
    hour = time.hour

    # Night hours : 12 AM to 5:59 AM
    if 0 <= hour < 6:
        night_messages[sender] += 1

# Calculate percentage
night_owl_score = {}

for person in participants:

    if total_messages[person] == 0:
        percentage = 0
    else:
        percentage = (night_messages[person] / total_messages[person]) * 100

    night_owl_score[person] = percentage

# Sort scores
sorted_night_owl = sorted(night_owl_score.items(),key=lambda x: x[1],reverse=True)

# Winner
night_owl = sorted_night_owl[0][0]
night_owl_value = sorted_night_owl[0][1]

# Runner-up
night_owl_runner = sorted_night_owl[1][0]
night_owl_runner_value = sorted_night_owl[1][1]

The story teller:

average number of words on each message per person calculated and sorted.

In [ ]:
story_words = {}
story_messages = {}

for person in participants:
    story_words[person] = 0
    story_messages[person] = 0

# Count total words and messages
for msg in messages:
    sender = msg["sender"]
    text = msg["text"].lower()

    # Skip media and deleted messages
    if text == "<media omitted>" or text == "this message was deleted":
        continue

    # Remove punctuation
    punctuation = ".,!?;:'\"()[]{}<>/-_*&%^$#@+=|\\`~"

    for ch in punctuation:
        text = text.replace(ch, " ")

    words = text.split()
    story_words[sender] += len(words)
    story_messages[sender] += 1

# average message length
storyteller_score = {}

for person in participants:

    if story_messages[person] == 0:
        average = 0
    else:
        average = story_words[person] / story_messages[person]

    storyteller_score[person] = average

# sort scores
sorted_storyteller = sorted(storyteller_score.items(),key=lambda x: x[1],reverse=True)

# winner
storyteller = sorted_storyteller[0][0]
storyteller_value = sorted_storyteller[0][1]

# runner-up
storyteller_runner = sorted_storyteller[1][0]
storyteller_runner_value = sorted_storyteller[1][1]

The Drama queen:

Calculates drama_score by counting '!','?' and uppercaste words are counted and sorted
Participant with highest score wins

In [ ]:
drama_queen_score = {}

for person in participants:
    drama_queen_score[person] = 0

# calculate drama score
for msg in messages:
    sender = msg["sender"]
    text = msg["text"]
    score = 0

    # count exclamation marks
    score += text.count("!")

    # count question marks
    score += text.count("?")

    # Count all caps words
    words = text.split()

    for word in words:

        if len(word) >= 3 and word.isupper():
            score += 2

    drama_queen_score[sender] += score

# Sort scores
sorted_drama_queen = sorted(drama_queen_score.items(),key=lambda x: x[1],reverse=True)

# Winner
drama_queen = sorted_drama_queen[0][0]
drama_queen_value = sorted_drama_queen[0][1]

# runner-up
drama_queen_runner = sorted_drama_queen[1][0]
drama_queen_runner_value = sorted_drama_queen[1][1]

The Ghost:

Counts total messages sent by each person sorts them in ascending order and finds out participant with lowest message count as the ghost.

In [ ]:
ghost_score = {}

for person in participants:
    ghost_score[person] = 0

# Count messages
for msg in messages:
    sender = msg["sender"]
    ghost_score[sender] += 1

# Sort scores (Lowest First)
sorted_ghost = sorted(ghost_score.items(),key=lambda x: x[1])

# Winner (Least messages)
ghost = sorted_ghost[0][0]
ghost_value = sorted_ghost[0][1]

# Runner-up
ghost_runner = sorted_ghost[1][0]
ghost_runner_value = sorted_ghost[1][1]

The Comedian:

Calculates percent of messages containin laughter words for each participant. Participant with highest percent wins

In [ ]:
laughter_words = [
    "lol",
    "lmao",
    "haha",
    "rofl",
    "lmfao"
]

comedy_messages = {}
total_messages = {}

for person in participants:
    comedy_messages[person] = 0
    total_messages[person] = 0

# Count messages containing laughter
for msg in messages:
    sender = msg["sender"]
    text = msg["text"].lower()
    total_messages[sender] += 1

    for word in laughter_words:

        if word in text:
            comedy_messages[sender] += 1
            break

# Calculate percentage
comedian_score = {}

for person in participants:

    if total_messages[person] == 0:
        percentage = 0
    else:
        percentage = (comedy_messages[person] / total_messages[person]) * 100

    comedian_score[person] = percentage

# Sort scores
sorted_comedian = sorted(comedian_score.items(),key=lambda x: x[1],reverse=True)

# Winner
comedian = sorted_comedian[0][0]
comedian_value = sorted_comedian[0][1]

# Runner-up
comedian_runner = sorted_comedian[1][0]
comedian_runner_value = sorted_comedian[1][1]

The Question Master:

Finds out question master by calculating the percentage of messages ending with '?' for each participant.
Highest percentage participant wins

In [ ]:
question_messages = {}
total_messages = {}

for person in participants:
    question_messages[person] = 0
    total_messages[person] = 0

# count question messages
for msg in messages:
    sender = msg["sender"]
    text = msg["text"].strip()
    total_messages[sender] += 1

    if text.endswith("?"):
        question_messages[sender] += 1

# calculate percentage
question_master_score = {}

for person in participants:

    if total_messages[person] == 0:
        percentage = 0
    else:
        percentage = (question_messages[person] / total_messages[person]) * 100

    question_master_score[person] = percentage

# sort scores
sorted_question_master = sorted(question_master_score.items(),key=lambda x: x[1],reverse=True)

# Winner
question_master = sorted_question_master[0][0]
question_master_value = sorted_question_master[0][1]

# runner-up
question_master_runner = sorted_question_master[1][0]
question_master_runner_value = sorted_question_master[1][1]

Bonus archetype - Deleteholic (most number of deleted message)

Counts deleted message for every participant is stored as score and is sorted. Highest score member wins.

In [ ]:
deleteholic_score = {}

for person in participants:
    deleteholic_score[person] = 0

# Count deleted messages
for msg in messages:
    sender = msg["sender"]
    text = msg["text"].strip().lower()

    if text == "this message was deleted":
        deleteholic_score[sender] += 1

# Sort scores
sorted_deleteholic = sorted(deleteholic_score.items(),key=lambda x: x[1],reverse=True)

# Winner
deleteholic = sorted_deleteholic[0][0]
deleteholic_value = sorted_deleteholic[0][1]

# Runner-up
deleteholic_runner = sorted_deleteholic[1][0]
deleteholic_runner_value = sorted_deleteholic[1][1]

In [ ]:
print("=" * 70)
print("PERSONALITY ARCHETYPES")
print("=" * 70)

print("\n1. THE SPAMMER")
print(f"Winner     : {spammer:<15} ({spammer_value:.2f} messages/burst)")
print(f"Runner-up  : {spammer_runner:<15} ({spammer_runner_value:.2f} messages/burst)")

print("\n2. THE GROUP MOM")
print(f"Winner     : {group_mom:<15} ({group_mom_value} caring words)")
print(f"Runner-up  : {group_mom_runner:<15} ({group_mom_runner_value} caring words)")

print("\n3. THE NIGHT OWL")
print(f"Winner     : {night_owl:<15} ({night_owl_value:.2f}% night messages)")
print(f"Runner-up  : {night_owl_runner:<15} ({night_owl_runner_value:.2f}% night messages)")

print("\n4. THE STORYTELLER")
print(f"Winner     : {storyteller:<15} ({storyteller_value:.2f} words/message)")
print(f"Runner-up  : {storyteller_runner:<15} ({storyteller_runner_value:.2f} words/message)")

print("\n5. THE DRAMA QUEEN")
print(f"Winner     : {drama_queen:<15} ({drama_queen_value} drama score)")
print(f"Runner-up  : {drama_queen_runner:<15} ({drama_queen_runner_value} drama score)")

print("\n6. THE GHOST")
print(f"Winner     : {ghost:<15} ({ghost_value} messages)")
print(f"Runner-up  : {ghost_runner:<15} ({ghost_runner_value} messages)")

print("\n7. THE COMEDIAN")
print(f"Winner     : {comedian:<15} ({comedian_value:.2f}% funny messages)")
print(f"Runner-up  : {comedian_runner:<15} ({comedian_runner_value:.2f}% funny messages)")

print("\n8. THE QUESTION MASTER")
print(f"Winner     : {question_master:<15} ({question_master_value:.2f}% question messages)")
print(f"Runner-up  : {question_master_runner:<15} ({question_master_runner_value:.2f}% question messages)")

print("\n9. BONUS ARCHETYPE : THE DELETEHOLIC")
print(f"Winner     : {deleteholic:<15} ({deleteholic_value} deleted messages)")
print(f"Runner-up  : {deleteholic_runner:<15} ({deleteholic_runner_value} deleted messages)")

print("=" * 70)

PERSONALITY ARCHETYPES

1. THE SPAMMER
Winner     : Rahul           (4.52 messages/burst)
Runner-up  : Aman            (2.72 messages/burst)

2. THE GROUP MOM
Winner     : Priya           (720 caring words)
Runner-up  : Karan           (101 caring words)

3. THE NIGHT OWL
Winner     : Aman            (68.37% night messages)
Runner-up  : Rahul           (8.81% night messages)

4. THE STORYTELLER
Winner     : Karan           (57.32 words/message)
Runner-up  : Aman            (5.39 words/message)

5. THE DRAMA QUEEN
Winner     : Neha            (3751 drama score)
Runner-up  : Priya           (250 drama score)

6. THE GHOST
Winner     : Vikas           (24 messages)
Runner-up  : Karan           (354 messages)

7. THE COMEDIAN
Winner     : Vikas           (16.67% funny messages)
Runner-up  : Rahul           (3.15% funny messages)

8. THE QUESTION MASTER
Winner     : Priya           (29.25% question messages)
Runner-up  : Aman            (6.53% question messages)

9. BONUS ARCHETYPE : THE DE

# Final Report

In [ ]:
group_name = input("Enter Group Name : ")
print("=" * 68)
print(f'{"GROUPDNA REPORT - " + group_name:^68}')
print(f"{total_days} days • {len(messages)} messages • {total_participants} members")
print("=" * 68)
print()
print(f"Period        : {start_date.strftime('%d %B %Y')} to {end_date.strftime('%d %B %Y')}")
print(f"Busiest Day   : {busiest_day} ({busiest_day_messages} messages)")
print(f"Busiest Hour  : {busiest_hour}:00 - {busiest_hour+1}:00")

print("\n" + "="*68)
print("MESSAGES PER PERSON")
print("="*68)

for person, count in sorted_people:
    bar = "\u2588" * int((count/max(message_count.values()))*20)
    percent = count/len(messages)*100
    print(f"{person:<10} {bar:<20} {count:>4} ({percent:.1f}%)")

print("\n" + "="*68)
print("ACTIVITY HEATMAP")
print("="*68)

print("          00 03 06 09 12 15 18 21")

for i in range(len(people)):
    print(f"{people[i]:<10}", end="")

    row = heatmap[i]
    row_max = max(row)

    if row_max == 0:
        row_max = 1

    for hour in [0,3,6,9,12,15,18,21]:

        value = row[hour]
        ratio = value/row_max

        if value == 0:
            symbol="."
        elif ratio<=0.25:
            symbol="\u2591"
        elif ratio<=0.50:
            symbol="\u2592"
        elif ratio<=0.75:
            symbol="\u2593"
        else:
            symbol="\u2588"

        print(f" {symbol} ",end="")

    print()

print("\n"+"="*68)
print("THIS GROUP'S FAVOURITE WORDS")
print("="*68)

for word,count in top_words[:5]:
    bar="\u2588"*int((count/top_words[0][1])*20)
    print(f"{word:<12}{bar:<20}{count}")

print("\n"+"="*68)
print("RESPONSE PATTERNS")
print("="*68)

print(f"Fastest Replier : {fastest_person}")
print(f"Slowest Replier : {slowest_person}")

print("\nLONGEST SILENT STREAKS")

for person, days in sorted(longest_streaks.items(),key=lambda x:x[1],reverse=True):
    print(f"{person:<10}: {days} days")

print("\n"+"="*68)
print("PERSONALITY ARCHETYPES")
print("="*68)

print(f"{spammer:<10} -> THE SPAMMER")
print(f"{group_mom:<10} -> THE GROUP MOM")
print(f"{night_owl:<10} -> THE NIGHT OWL")
print(f"{storyteller:<10} -> THE STORYTELLER")
print(f"{drama_queen:<10} -> THE DRAMA QUEEN")
print(f"{ghost:<10} -> THE GHOST")
print(f"{comedian:<10} -> THE COMEDIAN")
print(f"{question_master:<10} -> THE QUESTION MASTER")
print(f"{deleteholic:<10} -> THE DELETEHOLIC")

print("\n"+"="*68)
print("Generated by GroupDNA • Built with Python + NumPy")
print("="*68)

Enter Group Name : Hostel bois 4ever
                GROUPDNA REPORT - Hostel bois 4ever                 
60 days • 3174 messages • 6 members

Period        : 01 April 2024 to 30 May 2024
Busiest Day   : 2024-05-04 (76 messages)
Busiest Hour  : 18:00 - 19:00

MESSAGES PER PERSON
Rahul      ████████████████████  953 (30.0%)
Priya      ███████████████       718 (22.6%)
Neha       █████████████         635 (20.0%)
Aman       ██████████            490 (15.4%)
Karan      ███████               354 (11.2%)
Vikas                             24 (0.8%)

ACTIVITY HEATMAP
          00 03 06 09 12 15 18 21
Aman       ▓  ▓  .  .  .  ░  ░  ░ 
Karan      .  .  .  ▒  █  ▓  ▓  ▒ 
Neha       .  .  ░  █  ▓  ░  █  ▒ 
Priya      .  .  ░  █  █  ▒  ▓  ▒ 
Rahul      ░  ░  ░  ░  ▓  ▓  █  █ 
Vikas      .  .  .  ▒  ▓  ▒  ▓  ▒ 

THIS GROUP'S FAVOURITE WORDS
guys        ████████████████████318
hai         ████████████████    268
everyone    ████████████        203
telling     ███████████         179
bhai        ███

# Reflection

This project has helped me to strengthen my understanding of Python. I faced many challenges, response patterns especially consumed a lot of my time and debugging obviously one other challenge.
Overall it was a very good experiance working on this project.